# Chapter 3 - Lab 7: Anthropic API - the no-tools baseline

**Goal**: make a **direct call to Claude** with the data embedded in the prompt and ask the model to do the arithmetic. No tools, no agent loop. This non-agentic baseline tells you when an agent is worth its extra cost.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **Anthropic API client** (`anthropic`) - `client.beta.messages.create`.
* **Claude** (`claude-opus-5-5` by default; set `CC_ANTHROPIC_MODEL` to change). Effort `low`: this is a short task.
* The actual Claude **Agent** SDK version is `bakeoff/claude_agent_sdk_agent.py`.

## 1. Install packages

In [ ]:
%pip install -q anthropic pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Build a tools-free prompt

In [ ]:
enhanced_prompt = f"""
{input_message}

Available support data (Q3 2026):
- ZX1 (Zenith X1 Smartphone): tickets=2760, orders=18400
- FL-340 (FrostLine Refrigerator 340L): tickets=1134, orders=3150

Please:
1. Calculate the contact rate for both product lines (contact rate = tickets / orders x 100)
2. Compare them
3. Write a brief memo for the Head of Customer Care with one recommendation
"""

## 5. Call Claude

`fallbacks="default"` (beta) re-runs the request on another Claude model if a safety classifier declines it; we also check `stop_reason` before reading the text.

In [ ]:
import os
import anthropic

client = anthropic.Anthropic()
model = os.getenv("CC_ANTHROPIC_MODEL", "claude-opus-5-5")

response = client.beta.messages.create(
    model=model,
    max_tokens=16000,
    system=system_message,
    messages=[{"role": "user", "content": enhanced_prompt}],
    output_config={"effort": "low"},
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",
)

## 6. Read the response

In [ ]:
if response.stop_reason != "end_turn":
    raise RuntimeError(f"Incomplete baseline response: {response.stop_reason}")
memo = "\n".join(block.text for block in response.content if block.type == "text")
print(memo)
print(f"\nTokens: input={response.usage.input_tokens} output={response.usage.output_tokens}")

## 7. Results
You should get both contact rates (ZX1 = 15.0, FL-340 = 36.0) and a memo, with **no tool calls**.

* When the data fits in the prompt and the maths is simple, **no agent is needed**.
* Agents earn their extra cost (tokens, latency, code) when data is **live and private** (CRM, OMS),
  when the work needs **several steps or systems**, and when you need an **audit trail** of every lookup.